# Part 1, continued: evidence, wording and model type
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/advaith334/nst2046-memory-sycophancy/blob/main/notebooks/02_evidence_and_wording.ipynb)

Follow-up runs on 5 November 2026, designed after we saw the main results (post hoc):
- **Evidence passages:** a passage stating the answer, or a weak passage that only hints at it.
- **A false passage:** "An older reference states that ...", against a true memory.
- **Nine wordings** of the same memory, on Nemo and Mistral Small.
- **Base vs instruct:** Mistral Nemo base and instruct, run locally at the same quantisation.
- **MemSyco-Bench** (Xiang et al., 2026): an outside benchmark in free text.

In [ ]:
# Setup. In Colab this downloads the repo. On your own machine, start Jupyter from the repo folder.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("memsyco_lite"):
    !git clone -q https://github.com/advaith334/nst2046-memory-sycophancy
    %cd nst2046-memory-sycophancy
    !pip install --quiet openai
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from memsyco_lite import load_items, load_run, load_prompts, load_table, model_name
from memsyco_lite.data import show_prompt
from memsyco_lite.stats import effect, fmt, rate
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

## Evidence reduces the effect in steps
Nemo's rise in false answers with a stored false belief, by how much evidence the question carries.

In [ ]:
main = load_run("1-effect", "main")
items = load_items("longtail")
main = main[main.item.isin(items)]
runs = {"no passage": main, "weak passage": load_run("2-evidence", "weak-passage"), "passage states the answer": load_run("2-evidence", "strong-passage")}
pd.DataFrame([{"evidence": k, "Nemo": fmt(effect(v[v.model == "mistral-nemo"], "stored_incorrect", "no_belief")),
               "Mistral Small": fmt(effect(v[v.model == "mistral-small"], "stored_incorrect", "no_belief"))} for k, v in runs.items()])

In [ ]:
weak = load_run("2-evidence", "weak-passage")
print("Example weak passage:\n", items["lt_metal_melting"]["evidence_weak"])

## A false passage beats a true memory
Nemo and Small saw a passage giving the false answer, with no memory or with the true belief stored. The table shows how often each followed the passage (chose the false option).

In [ ]:
fp = load_run("2-evidence", "false-passage")
t = rate(fp)
t.index = [model_name(m) for m in t.index]
t

## Wording
The same false belief, worded nine ways (label off). Share of Nemo's and Small's replies choosing the false option (%).

In [ ]:
names = ["base", "notes", "may_be_wrong", "bare_fact", "said_once", "verified_profile", "label_short", "label_source", "label_check"]
rows = []
for n in names:
    w = load_run("2-evidence", f"wording-{n}")
    s = w[w.condition == "stored_incorrect"]
    rows.append({"wording": n, "Nemo": round(100 * s[s.model == "mistral-nemo"].false.mean()), "Mistral Small": round(100 * s[s.model == "mistral-small"].false.mean())})
pd.DataFrame(rows)

In [ ]:
# The memory block as each wording showed it, for one item
for n in ["base", "notes", "may_be_wrong", "bare_fact", "said_once", "verified_profile"]:
    w = load_run("2-evidence", f"wording-{n}"); p = load_prompts("2-evidence", f"wording-{n}")
    r = w[(w.item == "lt_metal_melting") & (w.condition == "stored_incorrect") & (w.rep == 1)].iloc[0]
    print(f"--- {n}\n{p[r.prompt_id][0]['content'].split(chr(10)+chr(10), 1)[-1]}\n")

The report's comparisons against the main wording, with intervals:

In [ ]:
load_table("2_wordings")

## Base vs instruct
MS1 predicted that instruction tuning causes the effect. Both models ran locally (llama.cpp, Q4_K_M) on the same plain-text prompt.

In [ ]:
base, inst = load_run("2-evidence", "base-model"), load_run("2-evidence", "instruct-model")
inst_plain = inst[inst.model == "nemo-instruct-local"]
print("Base:    ", fmt(effect(base, "stored_incorrect", "no_belief")))
print("Instruct:", fmt(effect(inst_plain, "stored_incorrect", "no_belief")))
# The report's table gives instruct +48: its analysis counted one reply that this quick re-read scores differently.
display(load_table("2_base_vs_instruct").query("measure == 'answers'")[["comparison", "base", "other", "est", "ci_low", "ci_high"]])
display(load_table("2_log_odds"))

## MemSyco-Bench
100 sampled free-text items, judged by DeepSeek V4 Flash as the benchmark does. We show the results tables only and do not redistribute the benchmark's items.

In [ ]:
display(load_table("2_memsyco_bench_rates")); display(load_table("2_memsyco_bench_tests"))

## What this shows
- Evidence cut Nemo's effect from +41 to +17 with a weak passage and to +1 with a passage stating the answer.
- A false passage beat a true memory: Nemo followed it 78% of the time without the memory and 81% with it.
- Wording alone moved Nemo between 25% and 79%, against 55% to 61% across five identical runs. Our headline +41 belongs to one wording.
- Base and instruct flipped equally (+50 and +48), so instruction tuning was not needed for deference in this pair.
- On MemSyco-Bench all five open models moved, including Llama, Gemma and Qwen.

**Limits.** All of this is post hoc, levels cross runs, and the base/instruct result is one quantised pair.

## Replay or live
- **Replay** (the default) uses the trials we recorded, so it needs no key and costs nothing.
- **Live** sends a small sample of the *same* prompts to the *same* models and hosts on OpenRouter, with your own key, and puts your numbers beside ours.

Put your key in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY`, or paste it when asked. Models answer at temperature 0.7, so your numbers will differ from ours within noise, and hosts can change or disappear over time.

In [ ]:
MODE = "replay"   # "replay": our recorded results, no key and no cost. "live": rerun a small sample with your own key.
CAP_USD = 0.50    # a live run stops once it has spent this much (US$)

Live mode reruns a sample of the weak-passage run. The base/instruct pair needs a local llama.cpp server and is replay only.

In [ ]:
if MODE == "live":
    from memsyco_lite.live import Client, get_key, sample, rerun_mcq, estimate_cost, compare
    recorded = load_run("2-evidence", "weak-passage")
    s = sample(recorded, models=['mistral-nemo', 'mistral-small'], conditions=['no_belief', 'stored_incorrect'], n_items=10)
    print(f"{len(s)} trials. The same trials cost us about US${estimate_cost(s):.4f}.")
    client = Client(get_key(), cap_usd=CAP_USD)
    live = rerun_mcq(client, s, "2-evidence", "weak-passage", load_items("longtail"))
    display(compare(s, live, "false"))
else:
    print("Replay mode. Set MODE = 'live' above to rerun a small sample with your own key.")